# Mission 6 — Autonomous Warp State Machine & Deliberation Loop

This notebook contains the integrated state machine, safe tool dispatcher, MDP trajectory evaluator, and cycle sentinel. It uses only the Python standard library. The historical transition matrix stays empty until a real historical dataset is supplied; no sample history is fabricated.

Run the implementation cell first, then the assert-based test suites at the end.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from hashlib import sha256
import json
import math
from typing import Any, Callable, Iterable, Mapping, Optional


class AgentState(Enum):
    IDLE = "IDLE"
    PERCEIVING = "PERCEIVING"
    DELIBERATING = "DELIBERATING"
    ACTING = "ACTING"
    OBSERVING = "OBSERVING"
    REFLECTING = "REFLECTING"
    HALTED = "HALTED"


LEGAL_TRANSITIONS = {
    AgentState.IDLE: {AgentState.PERCEIVING},
    AgentState.PERCEIVING: {AgentState.DELIBERATING},
    AgentState.DELIBERATING: {AgentState.ACTING},
    AgentState.ACTING: {AgentState.OBSERVING},
    AgentState.OBSERVING: {AgentState.REFLECTING},
    AgentState.REFLECTING: {AgentState.DELIBERATING, AgentState.HALTED},
    AgentState.HALTED: set(),
}


@dataclass
class Agent:
    """Deterministic finite-state agent with a hard transition budget."""
    max_steps: int = 20
    state: AgentState = AgentState.IDLE
    step_count: int = 0
    context: list[str] = field(default_factory=list)
    halt_reason: Optional[str] = None

    def __post_init__(self) -> None:
        if isinstance(self.max_steps, bool) or not isinstance(self.max_steps, int):
            raise TypeError("max_steps must be an integer")
        if not 0 <= self.max_steps <= 20:
            raise ValueError("max_steps must be between 0 and 20")
        if self.max_steps == 0:
            self.force_halt("Step budget is zero")

    def transition(self, new_state: AgentState) -> None:
        """Apply only an edge allowed by the formal transition graph."""
        if new_state not in LEGAL_TRANSITIONS[self.state]:
            raise ValueError(f"Illegal transition: {self.state.value} -> {new_state.value}")
        self.state = new_state

    def force_halt(self, reason: str) -> None:
        """Emergency/budget override; HALTED is terminal."""
        self.state = AgentState.HALTED
        self.halt_reason = reason
        self.context.append(f"HALTED: {reason}")

    def step(self, continue_deliberation: bool = True) -> AgentState:
        """Execute at most one legal transition; never exceed max_steps."""
        if self.state is AgentState.HALTED:
            return self.state
        if self.step_count >= self.max_steps:
            self.force_halt("Maximum step budget exceeded")
            return self.state

        if self.state is AgentState.REFLECTING:
            target = AgentState.DELIBERATING if continue_deliberation else AgentState.HALTED
        else:
            choices = LEGAL_TRANSITIONS[self.state]
            if len(choices) != 1:
                raise RuntimeError(f"A transition decision is required from {self.state.value}")
            target = next(iter(choices))
        self.transition(target)
        self.step_count += 1
        if self.state is AgentState.HALTED:
            self.halt_reason = self.halt_reason or "Normal terminal transition"
        elif self.step_count >= self.max_steps:
            self.force_halt("Maximum step budget reached")
        return self.state


def navigate_warp_vector(x: float, y: float, z: float) -> str:
    """Simulated navigation tool. Coordinates must be finite numbers."""
    coords = (x, y, z)
    if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in coords):
        raise ValueError("warp vector coordinates must be finite numbers")
    return f"Warp vector set to ({float(x):.3f}, {float(y):.3f}, {float(z):.3f})."


def adjust_plasma_shields(intensity: int) -> str:
    """Simulated shield tool."""
    if isinstance(intensity, bool) or not isinstance(intensity, int):
        raise ValueError("shield intensity must be an integer")
    if not 0 <= intensity <= 100:
        raise ValueError("shield intensity must be between 0 and 100")
    return f"Plasma shields adjusted to {intensity}% intensity."


def query_sensor_array(target: str) -> str:
    """Simulated sensor tool; does not access external systems."""
    if not isinstance(target, str) or not target.strip():
        raise ValueError("sensor target must be a non-empty string")
    return f"Sensor array scan queued for {target.strip()}."


# JSON Schema contracts use the standard Draft 2020-12 vocabulary subset needed
# here: type, properties, required, additionalProperties, minimum, maximum,
# and minLength. The validator fails closed on unknown schema keywords/types.
TOOL_SCHEMAS: dict[str, dict[str, Any]] = {
    "navigate_warp_vector": {
        "$schema": "https://json-schema.org/draft/2020-12/schema",
        "type": "object",
        "properties": {"x": {"type": "number"}, "y": {"type": "number"}, "z": {"type": "number"}},
        "required": ["x", "y", "z"], "additionalProperties": False,
    },
    "adjust_plasma_shields": {
        "$schema": "https://json-schema.org/draft/2020-12/schema",
        "type": "object",
        "properties": {"intensity": {"type": "integer", "minimum": 0, "maximum": 100}},
        "required": ["intensity"], "additionalProperties": False,
    },
    "query_sensor_array": {
        "$schema": "https://json-schema.org/draft/2020-12/schema",
        "type": "object",
        "properties": {"target": {"type": "string", "minLength": 1}},
        "required": ["target"], "additionalProperties": False,
    },
}

TOOL_FUNCTIONS: dict[str, Callable[..., str]] = {
    "navigate_warp_vector": navigate_warp_vector,
    "adjust_plasma_shields": adjust_plasma_shields,
    "query_sensor_array": query_sensor_array,
}


def _matches_json_type(value: Any, expected: str) -> bool:
    if expected == "object":
        return isinstance(value, dict)
    if expected == "array":
        return isinstance(value, list)
    if expected == "string":
        return isinstance(value, str)
    if expected == "boolean":
        return isinstance(value, bool)
    if expected == "integer":
        return isinstance(value, int) and not isinstance(value, bool)
    if expected == "number":
        return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)
    if expected == "null":
        return value is None
    raise ValueError(f"Unsupported JSON Schema type: {expected}")


def validate_json_schema(instance: Any, schema: Mapping[str, Any], path: str = "$", *, root: bool = True) -> None:
    """Validate the schema subset above; raise ValueError for any violation."""
    supported = {"$schema", "type", "properties", "required", "additionalProperties", "minimum", "maximum", "minLength"}
    unknown = set(schema) - supported
    if unknown:
        raise ValueError(f"Unsupported schema keyword(s): {sorted(unknown)}")
    expected = schema.get("type")
    if not isinstance(expected, str) or not _matches_json_type(instance, expected):
        raise ValueError(f"{path}: expected {expected!r}")
    if expected == "object":
        properties = schema.get("properties", {})
        required = schema.get("required", [])
        if not isinstance(properties, Mapping) or not isinstance(required, list):
            raise ValueError(f"{path}: malformed object schema")
        missing = [key for key in required if key not in instance]
        if missing:
            raise ValueError(f"{path}: missing required field(s): {', '.join(missing)}")
        if schema.get("additionalProperties", True) is False:
            extras = set(instance) - set(properties)
            if extras:
                raise ValueError(f"{path}: unexpected field(s): {', '.join(sorted(map(str, extras)))}")
        for key, value in instance.items():
            if key in properties:
                validate_json_schema(value, properties[key], f"{path}.{key}", root=False)
    if expected in ("number", "integer"):
        if "minimum" in schema and instance < schema["minimum"]:
            raise ValueError(f"{path}: must be >= {schema['minimum']}")
        if "maximum" in schema and instance > schema["maximum"]:
            raise ValueError(f"{path}: must be <= {schema['maximum']}")
    if expected == "string" and "minLength" in schema and len(instance) < schema["minLength"]:
        raise ValueError(f"{path}: must contain at least {schema['minLength']} character(s)")


def dispatch_action(payload: str | Mapping[str, Any], agent: Optional[Agent] = None) -> str:
    """Parse, validate, and safely dispatch one {tool, arguments} request."""
    try:
        request = json.loads(payload, parse_constant=lambda token: (_ for _ in ()).throw(ValueError(f"Invalid JSON constant: {token}"))) if isinstance(payload, str) else payload
        validate_json_schema(request, {
            "type": "object",
            "properties": {"tool": {"type": "string"}, "arguments": {"type": "object"}},
            "required": ["tool", "arguments"], "additionalProperties": False,
        })
        name = request["tool"]
        if name not in TOOL_SCHEMAS:
            raise ValueError(f"Unknown tool: {name}")
        validate_json_schema(request["arguments"], TOOL_SCHEMAS[name])
        result = TOOL_FUNCTIONS[name](**request["arguments"])
        observation = f"Tool Execution Result: {result}"
    except Exception as exc:
        observation = f"Tool Execution Error: {exc}"
    if agent is not None:
        agent.context.append(observation)
    return observation


@dataclass
class TrajectoryStep:
    state: Any
    action: Any
    reward: float
    next_state: Any


@dataclass
class TrajectoryLogger:
    records: list[TrajectoryStep] = field(default_factory=list)

    def log(self, state: Any, action: Any, reward: float, next_state: Any) -> None:
        if isinstance(reward, bool) or not isinstance(reward, (int, float)) or not math.isfinite(reward):
            raise ValueError("reward must be a finite number")
        self.records.append(TrajectoryStep(state, action, float(reward), next_state))

    def discounted_return(self, gamma: float = 0.95) -> float:
        return discounted_return([record.reward for record in self.records], gamma)


def discounted_return(rewards: Iterable[float], gamma: float = 0.95) -> float:
    """Compute G0 = sum_t gamma**t * r_t (gamma defaults to 0.95)."""
    if isinstance(gamma, bool) or not isinstance(gamma, (int, float)) or not 0 <= gamma <= 1:
        raise ValueError("gamma must be between 0 and 1")
    total = 0.0
    for t, reward in enumerate(rewards):
        if isinstance(reward, bool) or not isinstance(reward, (int, float)) or not math.isfinite(reward):
            raise ValueError("rewards must be finite numbers")
        total += (gamma ** t) * reward
    return total


def transition_frequency_matrix(historical_data: Optional[Iterable[Mapping[str, Any]]]) -> dict[tuple[Any, Any], dict[Any, float]]:
    """Estimate P(next_state | state, action) from supplied historical records.

    Each record must have state, action, and next_state keys. Returns {} when
    no dataset was supplied; it never invents historical observations.
    """
    if historical_data is None:
        return {}
    counts: dict[tuple[Any, Any], dict[Any, int]] = {}
    for row_number, record in enumerate(historical_data, start=1):
        if not isinstance(record, Mapping) or not {"state", "action", "next_state"} <= set(record):
            raise ValueError(f"historical row {row_number} must contain state, action, and next_state")
        key = (record["state"], record["action"])
        next_state = record["next_state"]
        row = counts.setdefault(key, {})
        row[next_state] = row.get(next_state, 0) + 1
    return {key: {s2: n / sum(row.values()) for s2, n in row.items()} for key, row in counts.items()}


def state_action_hash(environment_state: Any, action: Any) -> str:
    """Stable SHA-256 key for JSON-serializable state and action data."""
    try:
        canonical = json.dumps([environment_state, action], sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False)
    except (TypeError, ValueError) as exc:
        raise ValueError("state and action must be JSON-serializable finite data") from exc
    return sha256(canonical.encode("utf-8")).hexdigest()


def floyd_cycle_detect(start: Any, next_node: Callable[[Any], Optional[Any]], max_iterations: int = 1000) -> bool:
    """Floyd tortoise-and-hare cycle detection over a deterministic successor map."""
    if max_iterations <= 0:
        return False
    tortoise = next_node(start)
    hare = next_node(start)
    if hare is not None:
        hare = next_node(hare)
    iterations = 0
    while tortoise is not None and hare is not None and iterations < max_iterations:
        if tortoise == hare:
            return True
        tortoise = next_node(tortoise)
        hare = next_node(hare)
        if hare is not None:
            hare = next_node(hare)
        iterations += 1
    return False


@dataclass
class CycleSentinel:
    """Tracks state-action events and trips on a confirmed loop or 3 failures."""
    failure_limit: int = 3
    event_counts: dict[str, int] = field(default_factory=dict)
    next_event: dict[str, str] = field(default_factory=dict)
    previous_event: Optional[str] = None
    seen_events: set[str] = field(default_factory=set)
    critical_log: list[str] = field(default_factory=list)

    def observe(self, environment_state: Any, action: Any, agent: Agent, *, failed: bool = False) -> bool:
        key = state_action_hash(environment_state, action)
        self.event_counts[key] = self.event_counts.get(key, 0) + 1
        if self.previous_event is not None:
            self.next_event[self.previous_event] = key
        self.previous_event = key
        repeated_cycle = floyd_cycle_detect(key, lambda node: self.next_event.get(node), max_iterations=max(10, len(self.next_event) + 1))
        duplicate = key in self.seen_events
        self.seen_events.add(key)
        repeated_failed_action = failed and self.event_counts[key] >= self.failure_limit
        if repeated_cycle or repeated_failed_action:
            reason = "Repeated state-action cycle detected" if repeated_cycle else "Same failed state-action observed three times"
            message = f"CRITICAL SECURITY VIOLATION: {reason}; emergency tripwire activated."
            self.critical_log.append(message)
            agent.force_halt(message)
            return True
        return False


## Four required assert-based test suites

The test cell below checks the requested state machine and 21-call budget, safe invalid-payload handling, the specified discounted return, and Floyd cycle detection with a halt before step 10.


In [2]:
# Test Suite 1 — State machine, legal transitions, and strict 20-step ceiling
agent = Agent(max_steps=20)
assert agent.state is AgentState.IDLE
assert agent.step() is AgentState.PERCEIVING
try:
    agent.transition(AgentState.HALTED)  # forbidden by the normal transition graph
    raise AssertionError("Illegal transition was accepted")
except ValueError:
    pass
budget_agent = Agent(max_steps=20)
for _ in range(21):
    budget_agent.step()
assert budget_agent.state is AgentState.HALTED
assert budget_agent.step_count <= 20
assert budget_agent.halt_reason is not None
print("State Machine Test: PASS")


# Test Suite 2 — Invalid JSON/tool payload becomes an observation, without a crash
protocol_agent = Agent()
invalid_observation = dispatch_action('{"tool":"adjust_plasma_shields","arguments":{"intensity":-5}}', protocol_agent)
assert invalid_observation.startswith("Tool Execution Error: ")
assert protocol_agent.context[-1] == invalid_observation
assert "Tool Execution Error:" in dispatch_action("{invalid json")
assert dispatch_action('{"tool":"query_sensor_array","arguments":{"target":"nebula"}}').startswith("Tool Execution Result: ")
print("Protocol Dispatch Test: PASS")


# Test Suite 3 — MDP discounted return for the required rewards at gamma 0.95
required_rewards = [10, 0, -5, 20]
expected_return = 10 + (0.95 ** 1) * 0 + (0.95 ** 2) * (-5) + (0.95 ** 3) * 20
actual_return = discounted_return(required_rewards, gamma=0.95)
assert math.isclose(actual_return, expected_return, rel_tol=0.0, abs_tol=1e-12)
mdp_log = TrajectoryLogger()
for i, reward in enumerate(required_rewards):
    mdp_log.log(f"s{i}", f"a{i}", reward, f"s{i+1}")
assert math.isclose(mdp_log.discounted_return(), expected_return, rel_tol=0.0, abs_tol=1e-12)
assert transition_frequency_matrix(None) == {}  # no historical dataset is present
print(f"MDP Return Test: PASS (G0 = {actual_return:.5f})")


# Test Suite 4 — Floyd tortoise-and-hare cycle detection and emergency halt
mock_successors = {"s1": "s2", "s2": "s3", "s3": "s1"}
assert floyd_cycle_detect("s1", lambda state: mock_successors.get(state))
sentinel_agent = Agent(max_steps=20)
sentinel = CycleSentinel()
mock_events = ["s1", "s2", "s3", "s1", "s2", "s3"]
halted_before_step_10 = False
for step_number, state in enumerate(mock_events, start=1):
    # Each state is paired with the same mock action; the detector sees the
    # repeated s1/s2/s3 successor pattern and halts on the first repeated event.
    if sentinel.observe({"state": state}, {"action": "mock"}, sentinel_agent):
        halted_before_step_10 = step_number < 10
        break
assert halted_before_step_10
assert sentinel_agent.state is AgentState.HALTED
assert sentinel.critical_log
print("Sentinel Cycle Test: PASS (halted before step 10)")

print("All four Mission 6 test suites passed.")


State Machine Test: PASS
Protocol Dispatch Test: PASS
MDP Return Test: PASS (G0 = 22.63500)
Sentinel Cycle Test: PASS (halted before step 10)
All four Mission 6 test suites passed.
